#Fitting a third order polynomial to sine function.

Based on the [Learning PyTorch with Examples](https://colab.research.google.com/drive/1t3-yH6gWP4e3JYg27uiCpYabKvA9BI1q#scrollTo=uPwM_8H9G-yo&line=3&uniqifier=1) tutorial.

Warm-up: numpy
Before introducing PyTorch, we will first implement the network using numpy.

In [1]:
import math
import numpy as np

# Create random input and output data
x = np.linspace(-math.pi, math.pi, 2000)
y = np.sin(x)

# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

weights = [(a, b, c, d)]

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    # y = a + b x + c x^2 + d x^3
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = np.square(y_pred - y).sum()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

    weights.append((a, b, c, d))

print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

99 3590.9761229717933
199 2495.4675502857867
299 1736.581021706222
399 1210.3663163618414
499 845.1393724061611
599 591.413651947134
699 414.9904024174832
799 292.21135679951175
899 206.69345280283255
999 147.08041029213706
1099 105.49279972368142
1199 76.45845293312213
1299 56.173567262561875
1399 41.99172754929657
1499 32.07018240395634
1599 25.124735497459472
1699 20.259735394332118
1799 16.850043192821275
1899 14.459011464827608
1999 12.781436911274563
Result: y = -0.06317609314904078 + 0.8762300364279862 x + 0.0108989273229372 x^2 + -0.09610255708184517 x^3


Visualization (based on [PythonDataScienceHandbook](https://colab.research.google.com/github/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/04.14-Visualization-With-Seaborn.ipynb#scrollTo=f6FPDXYpMDgo))

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import animation

sns.set(font_scale=0.9)

# Create the background frame
fig = plt.figure(figsize=(12, 2))
ax1 = plt.subplot(1,2,1)

ax1.set_xlim((-math.pi, math.pi))
ax1.set_ylim((-1, 1))
ax1.set_xlabel("$x$")
ax1.set_ylabel("$y$")

txt_title = ax1.set_title("")
line1, = ax1.plot([], [], "b", lw=2)
line2, = ax1.plot([], [], "r", lw=2)

ax1.legend(["true", "pred"]);

# Hide the plain background
plt.close()

sampling_factor=5

def drawframe(t):
    a, b, c, d = weights[sampling_factor * t]
    y_true = np.sin(x)
    y_pred = a + b*x + c*x**2 + d*x**3
    line1.set_data(x, y_true)
    line2.set_data(x, y_pred)
    txt_title.set_text(f"Frame = {t*sampling_factor:4d}")
    return (line1, line2)

# blit=True re-draws only the parts that have changed.
anim = animation.FuncAnimation(fig, drawframe, frames=len(weights)//sampling_factor, interval=20, blit=True)

In [ ]:
from IPython.display import HTML
HTML(anim.to_html5_video())

PyTorch: Tensors

In [ ]:
import torch
import math

dtype = torch.float
# device = torch.device("cpu")
device = torch.device("cuda:0") # Uncomment this to run on GPU

# Create random input and output data
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum().item()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 1549.0316162109375
199 1028.262939453125
299 683.618896484375
399 455.52044677734375
499 304.54693603515625
599 204.61419677734375
699 138.4617156982422
799 94.66777038574219
899 65.67279052734375
999 46.47459411621094
1099 33.761756896972656
1199 25.342586517333984
1299 19.76644515991211
1399 16.072799682617188
1499 13.6258544921875
1599 12.004621505737305
1699 10.930341720581055
1799 10.21835994720459
1899 9.746444702148438
1999 9.43360424041748
Result: y = -0.005755438935011625 + 0.8331916332244873 x + 0.0009929107036441565 x^2 + -0.08998072147369385 x^3


PyTorch: Tensors + Autograd

In [ ]:
# -*- coding: utf-8 -*-
import torch
import math

dtype = torch.float
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x = torch.linspace(-math.pi, math.pi, 2000, dtype=dtype)
y = torch.sin(x)

# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss using operations on Tensors.
    # Now loss is a Tensor of shape (1,)
    # loss.item() gets the scalar value held in the loss.
    loss = (y_pred - y).pow(2).sum()
    if t % 100 == 99:
        print(t, loss.item())

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be Tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 192.47705078125
199 133.58270263671875
299 93.66865539550781
399 66.58847045898438
499 48.1954460144043
599 35.688934326171875
699 27.175567626953125
799 21.37378692626953
899 17.415334701538086
999 14.711565971374512
1099 12.862643241882324
1199 11.596843719482422
1299 10.729231834411621
1399 10.133896827697754
1499 9.724920272827148
1599 9.443648338317871
1699 9.249971389770508
1799 9.116495132446289
1899 9.024394989013672
1999 8.960774421691895
Result: y = 0.010601337999105453 + 0.8503444194793701 x + -0.0018289070576429367 x^2 + -0.0924205482006073 x^3


PyTorch: Tensors + Autograd + nn module

In [ ]:
# -*- coding: utf-8 -*-
import torch
import math


# Create Tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# For this example, the output y is a linear function of (x, x^2, x^3), so
# we can consider it as a linear layer neural network. Let's prepare the
# tensor (x, x^2, x^3).
p = torch.tensor([1, 2, 3]) # shape (3,)
xx = x.unsqueeze(-1).pow(p) # add a new dim at index -1 (last)

# In the above code, x.unsqueeze(-1) has shape (2000, 1), and p has shape
# (3,), for this case, broadcasting semantics will apply to obtain a tensor
# of shape (2000, 3)

# Use the nn package to define our model as a sequence of layers. nn.Sequential
# is a Module which contains other Modules, and applies them in sequence to
# produce its output. The Linear Module computes output from input using a
# linear function, and holds internal Tensors for its weight and bias.
# The Flatten layer flatens the output of the linear layer to a 1D tensor,
# to match the shape of `y`.
model = torch.nn.Sequential(
    torch.nn.Linear(3, 1), # inputdim = 3, outputdim=1: (2000, 3) -> (2000, 1)
    torch.nn.Flatten(0, 1) # from dim 0 to 1: (2000, 1) -> (2000,)
)

# The nn package also contains definitions of popular loss functions; in this
# case we will use Mean Squared Error (MSE) as our loss function.
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-6
for t in range(2000):

    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model(xx)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad

# You can access the first layer of `model` like accessing the first item of a list
linear_layer = model[0]

# For linear layer, its parameters are stored as `weight` and `bias`.
print(f'Result: y = {linear_layer.bias.item()} + {linear_layer.weight[:, 0].item()} x + {linear_layer.weight[:, 1].item()} x^2 + {linear_layer.weight[:, 2].item()} x^3')

99 1234.8671875
199 824.4723510742188
299 551.6575927734375
399 370.2388916015625
499 249.55429077148438
599 169.24073791503906
699 115.77218627929688
799 80.1603012084961
899 56.4307746887207
999 40.611656188964844
1099 30.060691833496094
1199 23.01974868774414
1299 18.318464279174805
1399 15.177650451660156
1499 13.078049659729004
1599 11.67359733581543
1699 10.733542442321777
1799 10.103874206542969
1899 9.681787490844727
1999 9.398662567138672
Result: y = 0.012914850376546383 + 0.8365218043327332 x + -0.0022280269768089056 x^2 + -0.09045440703630447 x^3
